# Notebook 3: Gender Inference

Assign gender to each author using the `gender-guesser` library, then compute article-level gender metrics.

**Paper target:** 83.3% male authorship, 16.7% female

**Outputs:**
- `data/processed/authorships_gendered.parquet`
- `data/processed/articles_gendered.parquet`

In [ ]:
import pandas as pd
import gender_guesser.detector as gender
from tqdm.notebook import tqdm
import sys
sys.path.insert(0, ".")
from utils import extract_first_name

# Load data
articles_df = pd.read_parquet("data/raw/articles.parquet")
authorships_df = pd.read_parquet("data/raw/authorships.parquet")

print(f"Articles: {len(articles_df):,}")
print(f"Authorships: {len(authorships_df):,}")

## 1. Extract First Names and Infer Gender

In [ ]:
# Extract first names
authorships_df["first_name"] = authorships_df["display_name"].apply(extract_first_name)

# Initialize gender detector
detector = gender.Detector(case_sensitive=False)

# Get gender for each unique first name (much faster than per-row)
unique_names = authorships_df["first_name"].dropna().unique()
print(f"Unique first names to classify: {len(unique_names):,}")

gender_map = {}
for name in tqdm(unique_names, desc="Inferring gender"):
    gender_map[name] = detector.get_gender(name)

# Map raw gender-guesser categories to simplified categories
GENDER_MAPPING = {
    "male": "male",
    "mostly_male": "male",
    "female": "female",
    "mostly_female": "female",
    "andy": "unclassified",       # androgynous
    "unknown": "unclassified",
}

# Apply to authorships
authorships_df["gender_raw"] = authorships_df["first_name"].map(gender_map)
authorships_df["gender"] = authorships_df["gender_raw"].map(GENDER_MAPPING).fillna("unclassified")

# Report classification results
print("\nRaw gender-guesser distribution:")
print(authorships_df["gender_raw"].value_counts())
print(f"\nSimplified distribution:")
print(authorships_df["gender"].value_counts())
print(f"\nClassification rate: {(authorships_df['gender'] != 'unclassified').mean():.1%}")

## 2. Compute Article-Level Gender Metrics

In [ ]:
# Compute article-level gender summary
def compute_article_gender(group):
    """Compute gender metrics for one article's authors."""
    classified = group[group["gender"] != "unclassified"]
    n_male = (classified["gender"] == "male").sum()
    n_female = (classified["gender"] == "female").sum()
    n_classified = n_male + n_female
    
    # First and last author gender
    sorted_auth = group.copy()
    position_order = {"first": 0, "middle": 1, "last": 2}
    sorted_auth["pos_order"] = sorted_auth["author_position"].map(position_order).fillna(1)
    sorted_auth = sorted_auth.sort_values("pos_order")
    
    first_gender = sorted_auth.iloc[0]["gender"] if len(sorted_auth) > 0 else "unclassified"
    last_gender = sorted_auth.iloc[-1]["gender"] if len(sorted_auth) > 1 else first_gender
    
    return pd.Series({
        "n_authors": len(group),
        "n_classified": n_classified,
        "n_male": n_male,
        "n_female": n_female,
        "n_unclassified": len(group) - n_classified,
        "pct_female": n_female / n_classified if n_classified > 0 else None,
        "pct_male": n_male / n_classified if n_classified > 0 else None,
        "has_female_author": n_female > 0,
        "all_female": n_female == n_classified and n_classified > 0,
        "first_author_gender": first_gender,
        "last_author_gender": last_gender,
    })

print("Computing article-level gender metrics...")
article_gender = authorships_df.groupby("work_id").apply(compute_article_gender).reset_index()

# Merge with article metadata
articles_gendered = articles_df.merge(article_gender, on="work_id", how="left")

# Also merge journal rankings
journal_list = pd.read_csv("data/processed/journal_list.csv")
articles_gendered = articles_gendered.merge(
    journal_list[["source_id", "abdc_rank"]], on="source_id", how="left"
)

print(f"\nArticles with gender data: {articles_gendered['pct_female'].notna().sum():,}")
print(f"\nOverall gender split (excluding unclassified):")
classified = articles_gendered[articles_gendered["pct_female"].notna()]
mean_pct_male = classified["pct_male"].mean()
mean_pct_female = classified["pct_female"].mean()
print(f"  Male:   {mean_pct_male:.1%}")
print(f"  Female: {mean_pct_female:.1%}")
print(f"\nPaper reference: Male 83.3%, Female 16.7%")

## 3. Save Results

In [ ]:
# Save
authorships_df.to_parquet("data/processed/authorships_gendered.parquet", index=False)
articles_gendered.to_parquet("data/processed/articles_gendered.parquet", index=False)

print("Saved:")
print(f"  data/processed/authorships_gendered.parquet ({len(authorships_df):,} rows)")
print(f"  data/processed/articles_gendered.parquet ({len(articles_gendered):,} rows)")

# Quick breakdown by ABDC rank
print("\nFemale authorship by journal rank:")
rank_gender = classified.groupby("abdc_rank")["pct_female"].mean().sort_index()
for rank, pct in rank_gender.items():
    print(f"  {rank}: {pct:.1%} female")
print("\nPaper finding: Male dominance increases with journal prestige")